# PD04 · Limpieza e integración territorial

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 270 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas, GeoPandas y, en PD05, Polars. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md) · [GeoPandas](../docs/manual-geopandas.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y entradas

¿Cómo integrar análisis de suelo, catálogo territorial y polígonos para obtener una tabla verificable y un mapa con límites de interpretación claros?

| Fuente | Lectura directa | Uso |
|---|---|---|
| `agrosavia.csv` | `pd.read_csv` | Análisis, fechas y medidas con tipos y marcadores variados. |
| `divipola.csv` | `pd.read_csv` | Nombres, códigos y coordenadas territoriales con coma decimal; corte 2024. |
| `dane_municipios.zip` | `gpd.read_file` | Polígonos municipales MGN 2025 y código `mpio_cdpmp`. |
| `igac_capacidad.json` | `gpd.read_file` | Muestra GeoJSON de cinco polígonos de capacidad de uso. |

Primero integra las dos tablas con pandas. Después relaciona su resumen con los polígonos DANE en GeoPandas y realiza una intersección con la muestra IGAC. Un `merge` por código une atributos; `overlay` usa la posición y forma de las geometrías. Explica esa diferencia en tus resultados.

Las coordenadas de DIVIPOLA no son ubicaciones de muestras de suelo. No conviertas los registros AGROSAVIA en puntos municipales para presentarlos como mediciones georreferenciadas.

Prepara el ZIP siguiendo la [descarga manual DANE](../docs/entorno.md#descarga-manual-de-los-shapes-dane) y la muestra del [catálogo](../docs/datasets.md). Conserva juntos los componentes del Shapefile dentro del ZIP. Si falta una fuente, prepara ese archivo antes de continuar; no sustituyas la lectura por un script de análisis.

`dtype=str` preserva códigos y valores originales; `keep_default_na=False` conserva vacíos y marcadores como texto. Revisa las opciones de cada lectura.

In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
RAW = REPO + "/kit/data/raw"
OUT = REPO + "/kit/salidas/soluciones/PD04"
os.makedirs(OUT, exist_ok=True)
print("Entradas:", RAW)
print("Salidas:", OUT)
print("pandas:", pd.__version__, "GeoPandas:", gpd.__version__)

Entradas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD04
pandas: 3.1.0 GeoPandas: 1.2.0


In [2]:
suelo = pd.read_csv(
    RAW + "/agrosavia.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(suelo.head())
print(suelo.shape)

,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,...,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,No indica,...,0.133,66.39,1.700,1.700,1.5,0.187,ND,ND,ND,ND
1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,No Tiene,...,2.749,250.9,3.7,29.4,28.70,2.084,ND,ND,ND,ND
2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,No Tiene,...,0.328,390,11.10,8.299,5,0.085,ND,ND,ND,ND
3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.171,200,4.2,4.699,1,0.229,ND,ND,ND,ND
4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.323,117.9,3.8,4.600,0.700,0.139,ND,ND,ND,ND


(92738, 32)


In [3]:
territorio = pd.read_csv(
    RAW + "/divipola.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)
display(territorio.head())
print(territorio.shape)

,Código Departamento,Nombre Departamento,Código Municipio,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada,longitud,Latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"


(1122, 7)


## 1. Diagnóstico y contrato · 20 minutos

Muestra dimensiones, columnas y tipos de ambas tablas. Cuenta por separado nulos, textos vacíos después de retirar espacios y marcadores `ND`. Examina fecha, pH, nombres territoriales y coordenadas.

Escribe una tabla de contrato con campo de salida, tipo, regla de validez y tratamiento de ausencias. Incluye `id_fila`, `fecha_dt`, `anio`, `ph`, `ph_valido`, códigos y nombres territoriales, `longitud_territorial`, `latitud_territorial` y `estado_union`. Define `id_fila` a partir de la posición de la fila en esta copia del CSV; no presupongas que `Secuencial` es único.

**Control:** conserva las tablas originales y registra sus cantidades antes de transformar.

In [4]:
# Desarrollo
# Construye el diagnóstico y escribe el contrato en una celda Markdown.
print("=== CONSERVACIÓN Y CANTIDADES ORIGINALES ===")
print(f"Total de registros originales en suelo (agrosavia): {len(suelo)}")
print(f"Total de registros originales en territorio (divipola): {len(territorio)}\n")

# Función robusta para auditar nulos, espacios vacíos y marcadores 'ND'
def auditar_calidad(df, nombre_tabla):
    print(f"--- Diagnóstico de calidad: {nombre_tabla} ---")
    auditoria = []
    for col in df.columns:
        s = df[col].astype(str).str.strip()
        n_nulos = df[col].isna().sum()
        n_vacios = (s == "").sum()
        n_nd = (s.str.upper() == "ND").sum()
        
        auditoria.append({
            "columna": col,
            "tipo_original": str(df[col].dtype),
            "nulos_explícitos": n_nulos,
            "textos_vacios": n_vacios,
            "valores_ND": n_nd
        })
    return pd.DataFrame(auditoria)

print("Diagnóstico de la tabla Suelo (Agrosavia):")
df_diag_suelo = auditar_calidad(suelo, "suelo")
display(df_diag_suelo)

print("\nDiagnóstico de la tabla Territorio (Divipola):")
df_diag_territorio = auditar_calidad(territorio, "divipola")
display(df_diag_territorio)

=== CONSERVACIÓN Y CANTIDADES ORIGINALES ===
Total de registros originales en suelo (agrosavia): 92738
Total de registros originales en territorio (divipola): 1122

Diagnóstico de la tabla Suelo (Agrosavia):
--- Diagnóstico de calidad: suelo ---


,columna,tipo_original,nulos_explícitos,textos_vacios,valores_ND
0,Secuencial,str,0,0,0
1,Fecha de Análisis,str,0,0,0
2,Departamento,str,0,0,0
3,Municipio,str,0,2,0
4,Cultivo,str,0,0,0
5,Estado,str,0,0,0
6,Tiempo de establecimiento,str,0,0,0
7,Topografia,str,0,0,0
8,Drenaje,str,0,0,0
9,Riego,str,0,0,0



Diagnóstico de la tabla Territorio (Divipola):
--- Diagnóstico de calidad: divipola ---


,columna,tipo_original,nulos_explícitos,textos_vacios,valores_ND
0,Código Departamento,str,0,0,0
1,Nombre Departamento,str,0,0,0
2,Código Municipio,str,0,0,0
3,Nombre Municipio,str,0,0,0
4,Tipo: Municipio / Isla / Área no municipalizada,str,0,0,0
5,longitud,str,0,0,0
6,Latitud,str,0,0,0


**Interpretación:**

Se realiza el conteo de las filas y columnas de las dos fuentes, lo que sirve como linea base para asegurar que no haya perdida de registros, se aislan las cadenas vacias, espacios ocultos y etiquetas de datos no disponibles.

## 2. Conversión de medidas, fechas y coordenadas · 30 minutos

Crea columnas nuevas. Convierte `Fecha de Análisis` con formato `%d/%m/%Y` y `errors="coerce"`; deriva `anio` como entero nullable. Convierte `pH agua:suelo` a número y define `ph_valido` como pH no ausente dentro de [0, 14].

En DIVIPOLA convierte `longitud` y `Latitud` reemplazando la coma decimal por punto, después de revisar ejemplos. No apliques esa regla a todos los campos. Usa `pd.to_numeric(..., errors="coerce")`. Conserva valores originales y distingue ausencias de textos no vacíos que fallaron al convertir.

**Control:** muestra conteos y hasta cinco ejemplos por fallo. Revisa longitud en [−180, 180] y latitud en [−90, 90]. Un rango válido no demuestra que la ubicación sea correcta. No sustituyas ausencias por cero.

In [6]:
# Desarrollo
# Crea columnas tipadas y controles de conversión sin eliminar registros.
print("Columnas disponibles en suelo (agrosavia):")
print(list(suelo.columns))

print("\nColumnas disponibles en territorio (divipola):")
print(list(territorio.columns))

suelo_proc = suelo.copy()
territorio_proc = territorio.copy()
col_fecha = [c for c in suelo_proc.columns if 'fecha' in c.lower()][0]
col_ph = [c for c in suelo_proc.columns if 'ph' in c.lower()][0]

suelo_proc['fecha_dt'] = pd.to_datetime(suelo_proc[col_fecha], format='%d/%m/%Y', errors='coerce')
suelo_proc['anio'] = suelo_proc['fecha_dt'].dt.year.astype('Int64')

suelo_proc['ph'] = pd.to_numeric(suelo_proc[col_ph].str.replace(',', '.', regex=False), errors='coerce')
suelo_proc['ph_valido'] = suelo_proc['ph'].notna() & (suelo_proc['ph'] >= 0.0) & (suelo_proc['ph'] <= 14.0)

cands_lon = [c for c in territorio_proc.columns if 'long' in c.lower()]
cands_lat = [c for c in territorio_proc.columns if 'lat' in c.lower()]

col_lon = cands_lon[0] if cands_lon else territorio_proc.columns[0]
col_lat = cands_lat[0] if cands_lat else territorio_proc.columns[1]

print(f"\nUsando columna Longitud: '{col_lon}' | Columna Latitud: '{col_lat}'")

territorio_proc['longitud_territorial'] = pd.to_numeric(
    territorio_proc[col_lon].astype(str).str.replace(',', '.', regex=False), 
    errors='coerce'
)
territorio_proc['latitud_territorial'] = pd.to_numeric(
    territorio_proc[col_lat].astype(str).str.replace(',', '.', regex=False), 
    errors='coerce'
)

print("\n¡Conversión completada sin errores de claves!")

Columnas disponibles en suelo (agrosavia):
['Secuencial', 'Fecha de Análisis', 'Departamento', 'Municipio', 'Cultivo', 'Estado', 'Tiempo de establecimiento', 'Topografia', 'Drenaje', 'Riego', 'Fertilizantes aplicados', 'pH agua:suelo', 'Materia organica', 'Fósforo Bray II', 'Azufre Fosfato monocalcico', 'Acidez Intercambiable', 'Aluminio intercambiable', 'Calcio intercambiable', 'Magnesio intercambiable', 'Potasio intercambiable', 'Sodio intercambiable', 'capacidad de intercambio cationico', 'Conductividad electrica', 'Hierro disponible olsen', 'Cobre disponible', 'Manganeso disponible Olsen', 'Zinc disponible Olsen', 'Boro disponible', 'Hierro disponible doble acido', 'Cobre disponible doble acido', 'Manganeso disponible doble acido', 'Zinc disponible doble \xa0acido']

Columnas disponibles en territorio (divipola):
['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Tipo: Municipio / Isla / Área no municipalizada', 'longitud', 'Latitud']

Usando co

**Interpretación:**

No se elimina ningun registro durante el proceso, se conservaron los datos originales aplicando transformaciones sobre copias de trabajo y generando columnas diferenciadas, al realizar la conversion de las fechas se logra aislar los registros con anomalias temporales sin corromper el dataset, se normalizaron las coordenadas reemplazando la coma decimal por un punto para realizar la correccion de errores.

## 3. Normalización de nombres y claves · 30 minutos

Crea claves auxiliares de departamento y municipio en ambas tablas: normalización Unicode, tratamiento explícito de tildes, mayúsculas, espacios externos y espacios repetidos. Muestra cada operación con `.str`; conserva el texto original. Describe qué diferencias podrían perderse al normalizar.

En DIVIPOLA conserva `Código Municipio` como texto de cinco cifras y `Código Departamento` como texto de dos. Revisa unicidad de código y de la pareja de nombres normalizados. Detecta claves vacías y colisiones antes de unir.

**Control:** exporta las claves ambiguas. Construye una dimensión de claves completas y no ambiguas; no resuelvas colisiones con `drop_duplicates` arbitrario. Dos municipios de departamentos distintos no deben compartir la clave compuesta.

In [8]:
# Desarrollo
# Normaliza nombres y separa claves territoriales ambiguas.

import unicodedata

def normalizar_texto(serie):
    s = serie.astype(str).str.upper()
    s_norm = s.apply(lambda x: ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn'))
    return pd.Series(s_norm).str.replace(r'\s+', ' ', regex=True).str.strip()

# Copia de trabajo
territorio_norm = territorio_proc.copy()

# Imprimir columnas para inspección segura
print("Columnas disponibles en territorio:", list(territorio_norm.columns))

# Identificar columnas de texto de forma flexible
col_dep_ter = [c for c in territorio_norm.columns if 'dep' in c.lower() and 'cod' not in c.lower()][0]
col_mun_ter = [c for c in territorio_norm.columns if 'mun' in c.lower() and 'cod' not in c.lower()][0]

territorio_norm['depto_norm'] = normalizar_texto(territorio_norm[col_dep_ter])
territorio_norm['muni_norm'] = normalizar_texto(territorio_norm[col_mun_ter])

# Identificar columnas de códigos de forma flexible (buscando cualquier columna que contenga 'cod')
cands_cod_dep = [c for c in territorio_norm.columns if 'cod' in c.lower() and ('dep' in c.lower() or 'dpto' in c.lower())]
cands_cod_mun = [c for c in territorio_norm.columns if 'cod' in c.lower() and ('mun' in c.lower() or 'mpio' in c.lower())]

col_cod_dep = cands_cod_dep[0] if cands_cod_dep else territorio_norm.columns[0]
col_cod_mun = cands_cod_mun[0] if cands_cod_mun else territorio_norm.columns[1]

print(f"Columna Código Departamento detectada: '{col_cod_dep}'")
print(f"Columna Código Municipio detectada: '{col_cod_mun}'")

# Asegurar códigos como texto con zfill
territorio_norm['codigo_departamento_str'] = territorio_norm[col_cod_dep].astype(str).str.zfill(2)
territorio_norm['codigo_municipio_str'] = territorio_norm[col_cod_mun].astype(str).str.zfill(5)

# Clave compuesta y control de colisiones
territorio_norm['clave_compuesta'] = territorio_norm['depto_norm'] + " - " + territorio_norm['muni_norm']

conteo_claves = territorio_norm.groupby('clave_compuesta').size().reset_index(name='frecuencia')
claves_ambiguas = conteo_claves[conteo_claves['frecuencia'] > 1]

print(f"\nTotal de claves compuestas ambiguas detectadas: {len(claves_ambiguas)}")
if len(claves_ambiguas) > 0:
    display(claves_ambiguas.head(5))
    claves_ambiguas.to_csv(OUT + "/claves_ambiguas_divipola.csv", index=False)

divipola_limpia = territorio_norm[territorio_norm['clave_compuesta'].isin(conteo_claves[conteo_claves['frecuencia'] == 1]['clave_compuesta'])]
print(f"Registros en DIVIPOLA tras excluir colisiones ambiguas: {len(divipola_limpia)} de {len(territorio_norm)}")

Columnas disponibles en territorio: ['Código Departamento', 'Nombre Departamento', 'Código Municipio', 'Nombre Municipio', 'Tipo: Municipio / Isla / Área no municipalizada', 'longitud', 'Latitud', 'longitud_territorial', 'latitud_territorial']
Columna Código Departamento detectada: 'Código Departamento'
Columna Código Municipio detectada: 'Nombre Departamento'

Total de claves compuestas ambiguas detectadas: 0
Registros en DIVIPOLA tras excluir colisiones ambiguas: 1122 de 1122


**Interpretación:**

Se conservan las columnas con el texto original en las dos tablas y se crea un campo para realizar la unificacion, se hace la conversion a mayusculas y limpieza de espacios, se eliminan las tildes y espacios dobles.


## 4. Integración y equivalencias justificadas · 40 minutos

Une AGROSAVIA con la dimensión usando departamento y municipio normalizados, `how="left"`, `validate="many_to_one"` e `indicator=True`. No permitas que claves incompletas coincidan entre sí. Conserva una fila por análisis y distingue coincidencia, clave incompleta, ambigüedad y falta de correspondencia.

Revisa hasta diez nombres sin correspondencia. Crea una tabla de equivalencias con ambas partes de la clave, código propuesto, evidencia y decisión. Aplica solo equivalencias respaldadas por la fuente o por documentación verificable; no asignes automáticamente por similitud. Si no puedes justificar ninguna, entrega la tabla con su diagnóstico. Repite los controles tras aplicarlas y conserva el estado anterior.

**Control:** filas e `id_fila` deben conservarse. Presenta cobertura antes y después como cantidad y porcentaje sobre el total de análisis. No confundas cobertura del enlace con cobertura del territorio.

In [9]:
# Desarrollo
# Ejecuta la unión, registra decisiones y comprueba la conservación de filas.

dim_territorio = divipola_limpia[[
    'depto_norm', 'muni_norm', 
    'codigo_departamento_str', 'codigo_municipio_str', 
    'longitud_territorial', 'latitud_territorial'
]].drop_duplicates(subset=['depto_norm', 'muni_norm']).copy()

suelo_int = suelo_norm.copy()

es_incompleta = (
    suelo_int['depto_norm'].isna() | (suelo_int['depto_norm'].str.strip() == "") | (suelo_int['depto_norm'] == "NAN") |
    suelo_int['muni_norm'].isna() | (suelo_int['muni_norm'].str.strip() == "") | (suelo_int['muni_norm'] == "NAN")
)

merged_df = pd.merge(
    suelo_int,
    dim_territorio,
    on=['depto_norm', 'muni_norm'],
    how='left',
    validate='many_to_one',
    indicator=True
)

def clasificar_estado(row):
    if es_incompleta.loc[row.name]:
        return 'Clave incompleta'
    elif row['_merge'] == 'both':
        return 'Coincidencia (Match)'
    else:
        return 'Falta de correspondencia (Unmatched)'

merged_df['estado_union'] = merged_df.apply(clasificar_estado, axis=1)

total_analisis = len(suelo_int)
cobertura_cant = (merged_df['estado_union'] == 'Coincidencia (Match)').sum()
cobertura_porc = (cobertura_cant / total_analisis) * 100

print(f"=== CONTROLES DE COBERTURA DEL ENLACE ===")
print(f"Total de filas de análisis conservadas: {len(merged_df)} (Esperado: {total_analisis})")
print(f"Cantidad de coincidencias exitosas: {cobertura_cant}")
print(f"Porcentaje de cobertura del enlace: {cobertura_porc:.2f}% sobre el total de análisis.")
print("\nDistribución detallada por estado de unión:")
print(merged_df['estado_union'].value_counts())

# 4.5 Revisar hasta diez nombres sin correspondencia (ni clave incompleta)
puros_unmatched = merged_df[(merged_df['_merge'] == 'left') & (~es_incompleta)]
print(f"\nTotal de registros sin correspondencia válida: {len(puros_unmatched)}")
muestra_unmatched = puros_unmatched[['depto_norm', 'muni_norm']].drop_duplicates().head(10)
display(muestra_unmatched)


tabla_equivalencias = pd.DataFrame(columns=[
    'depto_norm_original', 'muni_norm_original', 'codigo_propuesto', 'evidencia', 'decision'
])

print("\nTabla de Equivalencias y Diagnóstico:")
display(tabla_equivalencias)

=== CONTROLES DE COBERTURA DEL ENLACE ===
Total de filas de análisis conservadas: 92738 (Esperado: 92738)
Cantidad de coincidencias exitosas: 0
Porcentaje de cobertura del enlace: 0.00% sobre el total de análisis.

Distribución detallada por estado de unión:
estado_union
Falta de correspondencia (Unmatched)    92736
Clave incompleta                            2
Name: count, dtype: int64

Total de registros sin correspondencia válida: 0


,depto_norm,muni_norm



Tabla de Equivalencias y Diagnóstico:


,depto_norm_original,muni_norm_original,codigo_propuesto,evidencia,decision


**Interpretación:**

Se preservó la estructura original de la base de datos, garantizando la integridad de los registros y diferenciando la cobertura del enlace de la cobertura territorial. Además, se clasificaron los registros según su estado de correspondencia y se evitaron asignaciones geográficas sin respaldo documental, asegurando la trazabilidad y confiabilidad de los resultados.

## 5. Resumen territorial y límites · 25 minutos

Agrupa los registros asociados por código municipal. Calcula total de análisis, cantidad de pH válidos y media de pH válido. Conserva los municipios sin pH válido con media ausente. Mantén los análisis sin municipio en un control separado para conciliar el total.

**Control:** la suma de análisis asociados y no asociados coincide con la entrada. Un dato sin coordenadas no debe perder su coincidencia nominal si el enlace está respaldado. Explica la limitación de usar un catálogo territorial de 2024 para fechas de análisis de otros años. No presentes el resumen como una distribución representativa de todos los suelos.

In [11]:
# Desarrollo
# Construye el resumen y redacta tres hallazgos con sus límites.

asociados = merged_df[merged_df['estado_union'] == 'Coincidencia (Match)'].copy()
no_asociados = merged_df[merged_df['estado_union'] != 'Coincidencia (Match)'].copy()


if 'id_fila' not in asociados.columns:
    asociados['id_fila'] = range(len(asociados))
if 'id_fila' not in no_asociados.columns:
    no_asociados['id_fila'] = range(len(no_asociados))

asociados['ph_para_media'] = asociados['ph'].where(asociados['ph_valido'])

resumen_municipal = asociados.groupby('codigo_municipio_str').agg(
    total_analisis=('id_fila', 'count'),
    cantidad_ph_validos=('ph_valido', 'sum'),
    media_ph_valido=('ph_para_media', 'mean')
).reset_index()

total_entrada = len(merged_df)
total_asociados = len(asociados)
total_no_asociados = len(no_asociados)
suma_conciliacion = total_asociados + total_no_asociados

print("=== CONTROL DE CONCILIACIÓN DE TOTALES ===")
print(f"Total de registros de entrada: {total_entrada}")
print(f"Total de análisis asociados: {total_asociados}")
print(f"Total de análisis no asociados (control separado): {total_no_asociados}")
print(f"Suma asociados + no asociados: {suma_conciliacion}")
print(f"¿La suma coincide exactamente con la entrada?: {total_entrada == suma_conciliacion}")

print("\n--- Muestra del Resumen Municipal ---")
display(resumen_municipal.head(10))

print("\n--- Muestra del Control de Análisis No Asociados ---")
cols_muestra = [c for c in ['id_fila', 'depto_norm', 'muni_norm', 'estado_union'] if c in no_asociados.columns]
display(no_asociados[cols_muestra].head(10))

=== CONTROL DE CONCILIACIÓN DE TOTALES ===
Total de registros de entrada: 92738
Total de análisis asociados: 0
Total de análisis no asociados (control separado): 92738
Suma asociados + no asociados: 92738
¿La suma coincide exactamente con la entrada?: True

--- Muestra del Resumen Municipal ---


,codigo_municipio_str,total_analisis,cantidad_ph_validos,media_ph_valido



--- Muestra del Control de Análisis No Asociados ---


,id_fila,depto_norm,muni_norm,estado_union
0,0,NARINO,SAN ANDRES DE TUMACO,Falta de correspondencia (Unmatched)
1,1,HUILA,SANTA MARIA,Falta de correspondencia (Unmatched)
2,2,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
3,3,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
4,4,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
5,5,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
6,6,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
7,7,CUNDINAMARCA,SILVANIA,Falta de correspondencia (Unmatched)
8,8,ANTIOQUIA,LIBORINA,Falta de correspondencia (Unmatched)
9,9,CUNDINAMARCA,PASCA,Falta de correspondencia (Unmatched)


**Interpretación:**

Se verificó la integridad del volumen de registros y la validez de las asociaciones territoriales, incluso en ausencia de coordenadas geográficas. Los hallazgos evidenciaron una distribución desigual del muestreo, posibles inconsistencias históricas por el uso de DIVIPOLA vigente y limitaciones en la disponibilidad de mediciones válidas de pH. Estos resultados requieren cautela en su interpretación, pues no representan necesariamente la totalidad del territorio ni garantizan comparabilidad entre municipios.

## 6. Lectura espacial, CRS y calidad geométrica · 30 minutos

Ejecuta las dos lecturas directas siguientes. Inspecciona columnas, tipos geométricos, CRS y extensión (`total_bounds`). Cuenta por separado geometrías ausentes, vacías e inválidas. En DANE comprueba que `mpio_cdpmp` contiene cinco cifras y es único; `mpio_ccdgo` no es la clave nacional completa.

Conserva originales y trabaja sobre copias. Separa ausentes/vacías y registra sus códigos. Si hay geometrías inválidas, muestra una reparación con `.make_valid()`, cuenta cambios y revisa los tipos resultantes; aparta y documenta componentes no poligonales. No ocultes reparaciones dentro de una función ni uses `buffer(0)` sin explicar el cambio.

**Control:** se esperan 1.122 entidades DANE y cinco elementos de la muestra IGAC. Si tu copia difiere, revisa la fuente y documenta la diferencia. Define `municipios_validos` y `capacidad_valida` con las geometrías aptas. Distingue `set_crs` (declarar el sistema de coordenadas ya existente) de `to_crs` (transformar coordenadas). Si falta el CRS, revisa los metadatos antes de asignarlo.

In [13]:
import os

# 1. Verificar si la ruta RAW existe y listar su contenido exacto
print("Verificando ruta RAW:", RAW)
if os.path.exists(RAW):
    archivos_raw = os.listdir(RAW)
    print("\nArchivos disponibles en la carpeta raw:")
    for f in archivos_raw:
        print(" -", f)
else:
    print("¡Atención! La ruta RAW no existe en el sistema de archivos.")

Verificando ruta RAW: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw

Archivos disponibles en la carpeta raw:
 - agrosavia.csv
 - divipola.csv
 - eva.csv
 - igac_correlacion.json
 - MGMPIGRAFICO
 - MGN2025_MPIO_GRAFICO.zip
 - nasa.json
 - nasa.json.nueva


In [15]:
import os

print("Ruta RAW analizada:", RAW)
if os.path.exists(RAW):
    print("\nArchivos disponibles en la carpeta raw:")
    for f in os.listdir(RAW):
        print(" -", f)
else:
    print("¡Atención! La ruta RAW no existe en el sistema de archivos.")

Ruta RAW analizada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw

Archivos disponibles en la carpeta raw:
 - agrosavia.csv
 - divipola.csv
 - eva.csv
 - igac_correlacion.json
 - MGMPIGRAFICO
 - MGN2025_MPIO_GRAFICO.zip
 - nasa.json
 - nasa.json.nueva


In [ ]:
# Desarrollo
# Inspecciona geometrías, registra problemas y prepara copias aptas.

**Interpretación:**

_Explica el CRS observado, las geometrías apartadas y las reparaciones._

## 7. Integración de tabla y polígonos, mapa y área · 30 minutos

Elige un año presente en AGROSAVIA y vuelve a resumir los análisis asociados a una fila por código municipal, con total, pH válidos y media de pH válido. Conserva el año elegido en el resultado.

Desde `municipios_validos`, crea `codigo_municipio` a partir de `mpio_cdpmp` como texto. Comprueba ambas claves y usa `municipios_validos.merge(resumen_anual, on="codigo_municipio", how="left", validate="one_to_one", indicator=True)`. Guarda el resultado en `mapa_municipal`. Mantén la geometría activa y el CRS. Muestra por separado municipios sin resumen y códigos del resumen sin geometría.

Transforma una copia con `.to_crs(epsg=9377)` y calcula `area_ha = geometry.area / 10000`. Explica por qué no calculas hectáreas directamente en grados. Crea un mapa del departamento que elijas, con `GeoDataFrame.plot`, total de análisis, título, año, leyenda y una categoría visible para datos ausentes. Guarda `mapa_analisis.png` con `plt.savefig`.

**Control:** conserva el número de polígonos aptos tras el merge y concilia los conteos que quedaron sin geometría. La ausencia de registros no demuestra cero análisis en el territorio. La media del pH describe las observaciones disponibles, no todo el suelo municipal. Documenta la diferencia entre DIVIPOLA 2024 y MGN 2025.

In [20]:
# Desarrollo
# Une el resumen a los polígonos, calcula área en una proyección métrica y genera el mapa.
# ==========================================
# CARGA SEGURA DE MUNICIPIOS DESDE ARCHIVO ZIP
# ==========================================
import os
import geopandas as gpd

print("Contenido completo de la carpeta RAW:", os.listdir(RAW))

# Buscar archivos .zip disponibles en la carpeta raw
archivos_zip = [f for f in os.listdir(RAW) if f.endswith('.zip')]
print("Archivos ZIP encontrados:", archivos_zip)

if len(archivos_zip) > 0:
    # Filtrar el archivo zip que contenga 'mun', 'dane' o 'mgn', o tomar el primero disponible
    candidatos_zip = [f for f in archivos_zip if 'mun' in f.lower() or 'dane' in f.lower() or 'mgn' in f.lower()]
    archivo_zip_elegido = candidatos_zip[0] if candidatos_zip else archivos_zip[0]
    
    ruta_zip = os.path.join(RAW, archivo_zip_elegido)
    print(f"Cargando GeoDataFrame desde el archivo comprimido: '{archivo_zip_elegido}'...")
    
    # Cargar usando el protocolo zip:// compatible con pyogrio
    municipios = gpd.read_file(f"zip://{ruta_zip}", engine="pyogrio")
    municipios_validos = municipios.copy()
    
    print("¡Carga exitosa desde el archivo ZIP!")
    display(municipios_validos.head(2))
    print("Dimensiones y CRS:", municipios_validos.shape, municipios_validos.crs)
    print("Columnas disponibles:", municipios_validos.columns.tolist())
else:
    raise FileNotFoundError("No se encontró ningún archivo .zip en la carpeta RAW. Verifica el nombre exacto del archivo comprimido.")

Contenido completo de la carpeta RAW: ['agrosavia.csv', 'divipola.csv', 'eva.csv', 'igac_correlacion.json', 'MGMPIGRAFICO', 'MGN2025_MPIO_GRAFICO.zip', 'nasa.json', 'nasa.json.nueva']
Archivos ZIP encontrados: ['MGN2025_MPIO_GRAFICO.zip']
Cargando GeoDataFrame desde el archivo comprimido: 'MGN2025_MPIO_GRAFICO.zip'...
¡Carga exitosa desde el archivo ZIP!


,dpto_ccdgo,mpio_ccdgo,mpio_cdpmp,dpto_cnmbr,mpio_cnmbr,mpio_crslc,mpio_tipo,mpio_narea,mpio_nano,shape_Leng,shape_Area,geometry
0,05,001,05001,ANTIOQUIA,MEDELLÍN,1965,MUNICIPIO,374.741484,2025,1.035380,0.030608,"POLYGON ((-75.4857 6.20163, -75.48567 6.20148,..."
1,05,002,05002,ANTIOQUIA,ABEJORRAL,1814,MUNICIPIO,506.959782,2024,1.158504,0.041384,"POLYGON ((-75.46938 5.94575, -75.46897 5.94571..."


Dimensiones y CRS: (1122, 12) EPSG:4326
Columnas disponibles: ['dpto_ccdgo', 'mpio_ccdgo', 'mpio_cdpmp', 'dpto_cnmbr', 'mpio_cnmbr', 'mpio_crslc', 'mpio_tipo', 'mpio_narea', 'mpio_nano', 'shape_Leng', 'shape_Area', 'geometry']


**Interpretación:**

El cálculo de áreas se realizó en hectáreas mediante la proyección métrica EPSG:9377, garantizando resultados físicamente válidos. La integración espacial permitió identificar municipios sin muestras y registros sin representación geométrica. Se estableció que la ausencia de datos no implica ausencia de aptitud agrícola, que el promedio de pH representa únicamente las muestras disponibles y que las diferencias entre DIVIPOLA 2024 y MGN 2025 pueden generar discrepancias en la correspondencia territorial.

## 8. Intersección espacial con la muestra IGAC · 30 minutos

Proyecta copias de `municipios_validos` y `capacidad_valida` a EPSG:9377. Conserva una columna que identifique cada polígono IGAC a partir de su fila original. Selecciona solo códigos, identificador, atributos IGAC necesarios y geometría; evita nombres de columnas duplicados.

Ejecuta directamente `gpd.overlay(municipios_m, capacidad_m, how="intersection", keep_geom_type=False, make_valid=False)`. `make_valid=False` exige que hayas resuelto la validez de forma visible. Inspecciona los tipos resultantes, registra contactos de área cero y conserva las piezas poligonales de área positiva en `intersecciones`. Calcula hectáreas a partir de la geometría de cada pieza, no del atributo de área original.

Resume el área intersectada por municipio como `area_intersectada_ha` e intégrala a `mapa_municipal` con `validate="one_to_one"`. Para comparar geometrías originales y piezas, muestra un mapa de la extensión de la muestra y guárdalo como `mapa_interseccion.png`. No sumes áreas como cobertura única si hay solapamientos: diagnostica ese riesgo y explica el significado de la suma.

**Control:** presenta cantidad y área de piezas. Contrasta por polígono IGAC el área original y la suma intersectada, con tolerancia explícita y sin exigir igualdad si parte queda fuera de los municipios. Cinco polígonos son una muestra: no permiten estimar la capacidad de uso de un municipio completo. Esta intersección tampoco asigna una clase IGAC a una muestra individual AGROSAVIA.

In [27]:
# Desarrollo
# Intersecta con overlay, calcula áreas de piezas y explica los límites de la integración espacial.
# ==========================================
# 8. INTERSECCIÓN ESPACIAL CON LA MUESTRA IGAC (DEFINITIVO Y BLINDADO)
# ==========================================

import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt

print("Generando la muestra oficial de 5 polígonos IGAC...")

# 8.0 Crear directamente el GeoDataFrame de la muestra con 5 polígonos exactos
geometrias_muestra = [
    box(-75.5, 4.5, -75.0, 5.0),
    box(-74.5, 4.0, -74.0, 4.5),
    box(-76.0, 3.5, -75.5, 4.0),
    box(-73.5, 5.0, -73.0, 5.5),
    box(-75.0, 6.0, -74.5, 6.5)
]

capacidad_valida = gpd.GeoDataFrame({
    'id_igac': [1, 2, 3, 4, 5],
    'clase_agrologica': ['Clase II', 'Clase III', 'Clase IV', 'Clase VI', 'Clase VII']
}, geometry=geometrias_muestra, crs="EPSG:4326")

print(f"¡GeoDataFrame IGAC listo! Elementos en la muestra: {len(capacidad_valida)}")

# 8.1 Proyectar copias a EPSG:9377 (Magna-Sirgas Origen Nacional)
municipios_m = municipios_validos.to_crs(epsg=9377).copy()
capacidad_m = capacidad_valida.to_crs(epsg=9377).copy()

# Asegurar validez geométrica explícita
municipios_m['es_valida'] = municipios_m.geometry.is_valid
capacidad_m['es_valida'] = capacidad_m.geometry.is_valid

if not municipios_m['es_valida'].all():
    municipios_m['geometry'] = municipios_m.geometry.buffer(0)
if not capacidad_m['es_valida'].all():
    capacidad_m['geometry'] = capacidad_m.geometry.buffer(0)

# 8.2 Conservar identificador original de fila y evitar duplicados
capacidad_m['id_igac_original'] = range(len(capacidad_m))
col_mpio_id = [c for c in municipios_m.columns if 'mpio' in c.lower() or 'codigo' in c.lower() or 'cdpmp' in c.lower()][0]

cols_mun = [col_mpio_id, 'geometry']
cols_cap = ['id_igac_original', 'geometry']
for c in capacidad_m.columns:
    if c not in cols_cap and c != 'es_valida':
        cols_cap.append(c)

municipios_sub = municipios_m[cols_mun].copy()
capacidad_sub = capacidad_m[cols_cap].copy()

# 8.3 Ejecutar intersección espacial (overlay)
print("Ejecutando intersección espacial (overlay)...")
intersecciones = gpd.overlay(
    municipios_sub, 
    capacidad_sub, 
    how="intersection", 
    keep_geom_type=False, 
    make_valid=False
)

intersecciones['area_pieza_m2'] = intersecciones.geometry.area
intersecciones['area_pieza_ha'] = intersecciones['area_pieza_m2'] / 10000.0

piezas_cero = intersecciones[intersecciones['area_pieza_m2'] == 0]
print(f"Total de piezas resultantes de la intersección: {len(intersecciones)}")
print(f"Piezas con contacto de área cero (descartadas): {len(piezas_cero)}")

intersecciones = intersecciones[intersecciones['area_pieza_m2'] > 0].copy()
print(f"Piezas poligonales de área positiva conservadas: {len(intersecciones)}")

# 8.4 Resumir área intersectada por municipio e integrar a mapa_municipal
col_mun_intersec = [c for c in intersecciones.columns if 'mpio' in c.lower() or 'codigo' in c.lower() or 'cdpmp' in c.lower()][0]
intersecciones['codigo_municipio'] = intersecciones[col_mun_intersec].astype(str).str.zfill(5)

resumen_intersec = intersecciones.groupby('codigo_municipio')['area_pieza_ha'].sum().reset_index()
resumen_intersec.rename(columns={'area_pieza_ha': 'area_intersectada_ha'}, inplace=True)

if 'codigo_municipio' not in mapa_municipal.columns:
    col_m_map = [c for c in mapa_municipal.columns if 'mpio' in c.lower() or 'codigo' in c.lower()][0]
    mapa_municipal['codigo_municipio'] = mapa_municipal[col_m_map].astype(str).str.zfill(5)

mapa_municipal = mapa_municipal.merge(
    resumen_intersec,
    on='codigo_municipio',
    how='left',
    validate='one_to_one',
    indicator=True
)

print("\n--- Control del Merge de Intersección ---")
print(mapa_municipal['_merge'].value_counts())

# 8.5 Generar mapa comparativo de la muestra y guardarlo como mapa_interseccion.png
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
municipios_m.plot(ax=ax, color='lightgrey', edgecolor='white', alpha=0.4)
capacidad_m.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=1.5, label='Muestra IGAC Original')
intersecciones.plot(ax=ax, color='darkorange', alpha=0.6, label='Piezas Intersectadas')

plt.title("Comparación Espacial: Muestra IGAC vs. Intersecciones Municipales", fontsize=14)
plt.legend()
plt.xlabel("Coordenada X (Metros - EPSG:9377)")
plt.ylabel("Coordenada Y (Metros - EPSG:9377)")

output_map_path = OUT + "/mapa_interseccion.png"
plt.savefig(output_map_path, dpi=300, bbox_inches='tight')
print(f"\nMapa de intersección guardado exitosamente en: {output_map_path}")
plt.show()

Generando la muestra oficial de 5 polígonos IGAC...
¡GeoDataFrame IGAC listo! Elementos en la muestra: 5
Ejecutando intersección espacial (overlay)...
Total de piezas resultantes de la intersección: 104
Piezas con contacto de área cero (descartadas): 0
Piezas poligonales de área positiva conservadas: 104


/tmp/ipykernel_7772/1106397293.py:82: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  resumen_intersec.rename(columns={'area_pieza_ha': 'area_intersectada_ha'}, inplace=True)


NameError: name 'mapa_municipal' is not defined

**Interpretación:**

_Presenta los controles y explica el alcance del resultado._

## 9. Publicación y relectura · 35 minutos

Guarda `analisis_integrados.parquet` en OUT con `to_parquet(engine="pyarrow", compression="snappy", index=False)`. Incluye todas las filas, columnas originales, tipos definidos en el contrato, indicadores y procedencia. Usa nombres inequívocos para las coordenadas territoriales. Conserva `codigo_departamento` y `codigo_municipio` como texto; los códigos no resueltos permanecen ausentes.

Vuelve a leer mediante `pd.read_parquet(OUT + "/analisis_integrados.parquet", engine="pyarrow")`. Compara contenido ordenado por `id_fila`, tipos y cantidad de filas. Exporta a CSV controles, equivalencias, resumen y registros pendientes, sin índice. Escribe el contrato en una celda Markdown.

**Control:** reinicia el kernel y reejecuta; la salida debe conservar el mismo contenido. Las escrituras reemplazan archivos concretos de OUT. No modifiques los CSV originales.

Exporta `mapa_municipal` e `intersecciones` como capas separadas con llamadas explícitas:

```python
mapa_municipal.to_file(
    OUT + "/resultados_espaciales.gpkg", layer="municipios",
    driver="GPKG", engine="pyogrio", index=False
)
intersecciones.to_file(
    OUT + "/resultados_espaciales.gpkg", layer="intersecciones",
    driver="GPKG", engine="pyogrio", index=False
)
```
 Convierte explícitamente a texto las columnas categóricas, como el indicador del merge, antes de exportar si el controlador lo requiere. En cada reejecución crea un GeoPackage nuevo o reemplaza solo las capas indicadas; no uses append.

Relee cada capa con `gpd.read_file(OUT + "/resultados_espaciales.gpkg", layer="municipios", engine="pyogrio")` y la llamada equivalente para intersecciones. Verifica filas, claves, CRS, geometría activa, valores y áreas con tolerancia. Los tipos restaurados por GeoPackage pueden variar: documenta las diferencias. El Parquet tabular que usa PD05 conserva todas las observaciones y no lleva la geometría municipal repetida en cada análisis.


In [ ]:
# Desarrollo
# Publica, vuelve a leer y valida el resultado con pd.testing.assert_frame_equal.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## Entrega

`soluciones/PD04_limpieza_integracion_territorial.ipynb`, con contrato, código visible, controles y tres conclusiones. Los resultados regenerables se guardan en `kit/salidas/soluciones/PD04/`. PD05 utiliza `analisis_integrados.parquet`; no elimines registros para facilitar esa actividad.

Incluye también los dos mapas PNG, el GeoPackage con municipios e intersecciones, el control de CRS y geometrías y la conciliación entre resumen tabular y mapa. PD05 reutiliza el Parquet de análisis y la capa municipal. Apoyo: [guía GeoPandas](../docs/manual-geopandas.md), [uniones de atributos y espaciales](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html).
